In [1]:
import os
import rasterio
import numpy as np
from importlib_resources import files
from pathlib import Path
from rasterio.enums import Resampling
from tqdm import tqdm

from beak.utilities.raster_processing import unify_raster_grids
from beak.utilities.io import save_raster, create_file_list, check_path, create_file_folder_list, data_folder
from beak.utilities.preparation import impute_data


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [2]:
BASE_PATH = data_folder()
EPSG_CODE = 102008
RESOLUTION = 100

BASE_EXTENT = "tusk_great_basin_a2b"
BASE_RASTER = BASE_PATH / "processed" / str(f"regional_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "base_raster" / "template_raster_geodawn_area2b.tif"
BASE_EVIDENCE = "geophysics"
CMA_EXTENT = "regional"

# Export
# If some data sets are already **LOG** scaled they need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "unified" / BASE_EVIDENCE
PATH_EXPORT = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "unified_imputed" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified\geophysics
Export folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified_imputed\geophysics


Select subfolders to be scaled.

In [3]:
for folder in os.listdir(PATH_INPUT):
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


gravity
magnetics
magnetotellurics
radiometric
seismic


In [4]:
SELECTION = ["gravity", 
             "magnetics",
             "magnetotellurics",
             "radiometric",
             "seismic"]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified\geophysics\gravity
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified\geophysics\magnetics
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified\geophysics\magnetotellurics
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified\geophysics\radiometric
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2b_102008_100\unified\geophysics\seismic


**Select files**

In [5]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Show results
print(f"Found {len(file_list)} files.")


Found 15 files.


**Run unification**

In [6]:
base_raster = rasterio.open(BASE_RASTER)

for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    
    raster_array = raster.read()
    raster_array = np.where(raster_array == raster.nodata, np.nan, raster_array)
    raster_array = impute_data(raster_array)
    
    raster_array = np.where(np.isnan(raster_array), base_raster.nodata, raster_array)
    raster_array = np.where(base_raster.read() == base_raster.nodata, np.nan, raster_array)

    output_folder = OUT_FOLDER / str(folder_relative)
    out_path = output_folder / file.name
    check_path(Path(os.path.dirname(out_path)))
    save_raster(out_path, array=raster_array, dtype="float32", metadata=raster.meta, overwrite=True)


100%|██████████| 15/15 [00:07<00:00,  1.96it/s]
